데이터 수집 -> 정제/EDA -> 전처리 -> 데이터 분리(학습/테스트) -> 모델 선정 -> 학습 -> 평가 -> 피드백

In [28]:
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 폰트
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 수집

데이터 : PowerCo.ltd의 고객 이탈률
- https://www.kaggle.com/datasets/sharmadeepankaj/customer-churn-of-powerco-ltd?select=client_data.csv

### target = churn (향후 3개월 동안 고객 이탈 여부)
### price_data.csv : 시간의 흐름에 따른 전기 및 가스 가격
| 컬럼명(한글) | 컬럼명(영어) | 설명(한글) | 분포 | 데이터 타입 |
|---|---|---|---|---|
| 고객사 ID | id | 고객사(회사) 고유 식별자 | 16,096개 고객, 중복 존재 | object |
| 가격 기준일 | price_date | 가격이 적용되는 기준 날짜 | 2015-01-01 ~ 2015-12-01 (월 단위) | object |
| 비수기 에너지 변동가 | price_off_peak_var | 비수기(1구간) 에너지 변동 가격 | 0.10~0.16 집중, max ≈ 0.28 | float64 |
| 피크 에너지 변동가 | price_peak_var | 피크(2구간) 에너지 변동 가격 | 다수 0값, 유효값 0.08~0.12 | float64 |
| 중간피크 에너지 변동가 | price_mid_peak_var | 중간피크(3구간) 에너지 변동 가격 | 대부분 0, 일부 0.06~0.10 | float64 |
| 비수기 전력 고정가 | price_off_peak_fix | 비수기(1구간) 전력 고정 가격 | 40~46 집중, max ≈ 59.4 | float64 |
| 피크 전력 고정가 | price_peak_fix | 피크(2구간) 전력 고정 가격 | 다수 0, 일부 24~26 / 35~36 | float64 |
| 중간피크 전력 고정가 | price_mid_peak_fix | 중간피크(3구간) 전력 고정 가격 | 다수 0, 일부 16~17 | float64 |

### client_data.csv : 고객의 일반 정보
| 컬럼명(한글) | 컬럼명(영어) | 설명(한글) | 분포 | 데이터 타입 |
|---|---|---|---|---|
| 고객사 ID | id | 고객사 고유 식별자 | 14,606개 모두 고유값 | object |
| 판매 채널 | channel_sales | 고객이 유입된 판매 채널 코드 | 3개 주요 범주 (약 46% / 28% / 26%) | object |
| 연간 전기 소비량 | cons_12m | 지난 12개월간 전기 소비량 | 강한 우측 치우침, max ≈ 6.21M | int64 |
| 연간 가스 소비량 | cons_gas_12m | 지난 12개월간 가스 소비량 | 강한 우측 치우침, max ≈ 4.15M | int64 |
| 전월 전기 소비량 | cons_last_month | 지난달 전기 소비량 | 우측 치우침, max ≈ 771K | int64 |
| 계약 활성일 | date_activ | 계약이 시작된 날짜 | 2003-05-09 ~ 2014-09-01 | object |
| 계약 종료일 | date_end | 계약이 종료된 날짜 | 2016-01-28 ~ 2017-06-13 | object |
| 상품 수정일 | date_modif_prod | 상품이 마지막으로 수정된 날짜 | 2003-05-09 ~ 2016-01-29 | object |
| 계약 갱신일 | date_renewal | 다음 계약 갱신 예정일 | 2013-06-26 ~ 2016-01-28 | object |
| 예상 전기 소비량(12개월) | forecast_cons_12m | 향후 12개월 예상 전기 소비량 | 대부분 저값, max ≈ 82.9K | float64 |
| 예상 전기 소비량(1년) | forecast_cons_year | 향후 1년 예상 전기 소비량 | 대부분 저값, max ≈ 175K | int64 |
| 예상 에너지 할인 | forecast_discount_energy | 향후 적용될 할인 금액 | 0~30 구간 집중 | float64 |
| 예상 계량기 임대료 | forecast_meter_rent_12m | 향후 12개월 계량기 임대료 | 0~30 구간 집중 | float64 |
| 비수기 에너지 가격 | forecast_price_energy_off_peak | 비수기 에너지 단가 예측 | 0~600 구간 집중 | float64 |
| 피크 에너지 가격 | forecast_price_energy_peak | 피크 시간대 에너지 단가 예측 | 0.11~0.18 집중 | float64 |
| 비수기 전력 가격 | forecast_price_pow_off_peak | 비피크 시간대 전력 단가 | 0.08~0.12 집중 | float64 |
| 가스 고객 여부 | has_gas | 가스 상품 가입 여부 | True 18% / False 82% | object |
| 유료 소비량 | imp_cons | 실제 청구 기준 소비량 | 우측 치우침, max ≈ 15K | float64 |
| 전력 총마진 | margin_gross_pow_ele | 전력 구독 총 마진 | 0~375 범위 집중 | float64 |
| 전력 순마진 | margin_net_pow_ele | 전력 구독 순 마진 | 총마진과 유사 분포 | float64 |
| 활성 상품 수 | nb_prod_act | 활성화된 상품 및 서비스 수 | 1~2개 구간 집중 | int64 |
| 총 순마진 | net_margin | 고객 전체 순마진 | 극단적 우측 치우침 | float64 |
| 가입 연수 | num_years_antig | 고객 가입 후 경과 연수 | 2~7년 구간 집중 | int64 |
| 가입 캠페인 | origin_up | 최초 가입 캠페인 코드 | 3개 주요 그룹 | object |
| 계약 전력량 | pow_max | 계약된 최대 전력량 | 3~320 범위 | float64 |
| 이탈 여부 | churn | 향후 3개월 내 이탈 여부 | 1 약 10% / 0 약 90% | int64 |

In [29]:
price_df = pd.read_csv('./data/price_data.csv')
price_df

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.000000,0.000000,44.266931,0.00000,0.000000
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.000000,0.000000,44.266931,0.00000,0.000000
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.000000,0.000000,44.266931,0.00000,0.000000
...,...,...,...,...,...,...,...,...
192997,16f51cdc2baa19af0b940ee1b3dd17d5,2015-08-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555
192998,16f51cdc2baa19af0b940ee1b3dd17d5,2015-09-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555
192999,16f51cdc2baa19af0b940ee1b3dd17d5,2015-10-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555
193000,16f51cdc2baa19af0b940ee1b3dd17d5,2015-11-01,0.119916,0.102232,0.076257,40.728885,24.43733,16.291555


In [30]:
price_df.isnull().sum()

id                    0
price_date            0
price_off_peak_var    0
price_peak_var        0
price_mid_peak_var    0
price_off_peak_fix    0
price_peak_fix        0
price_mid_peak_fix    0
dtype: int64

In [31]:
client_df = pd.read_csv('./data/client_data.csv')
client_df

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,t,0.00,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,f,0.00,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,f,0.00,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,f,0.00,30.22,30.22,1,25.46,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,f,52.32,44.91,44.91,1,47.98,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14601,18463073fb097fc0ac5d3e040f356987,foosdfpfkusacimwkcsosbicdxkicaua,32270,47940,0,2012-05-24,2016-05-08,2015-05-08,2014-05-26,4648.01,...,t,0.00,27.88,27.88,2,381.77,4,lxidpiddsbxsbosboudacockeimpuepw,15.000,0
14602,d0a6f71671571ed83b2645d23af6de00,foosdfpfkusacimwkcsosbicdxkicaua,7223,0,181,2012-08-27,2016-08-27,2012-08-27,2015-08-28,631.69,...,f,15.94,0.00,0.00,1,90.34,3,lxidpiddsbxsbosboudacockeimpuepw,6.000,1
14603,10e6828ddd62cbcf687cb74928c4c2d2,foosdfpfkusacimwkcsosbicdxkicaua,1844,0,179,2012-02-08,2016-02-07,2012-02-08,2015-02-09,190.39,...,f,18.05,39.84,39.84,1,20.38,4,lxidpiddsbxsbosboudacockeimpuepw,15.935,1
14604,1cf20fd6206d7678d5bcafd28c53b4db,foosdfpfkusacimwkcsosbicdxkicaua,131,0,0,2012-08-30,2016-08-30,2012-08-30,2015-08-31,19.34,...,f,0.00,13.08,13.08,1,0.96,3,lxidpiddsbxsbosboudacockeimpuepw,11.000,0


In [32]:
client_df.isnull().sum()

id                                0
channel_sales                     0
cons_12m                          0
cons_gas_12m                      0
cons_last_month                   0
date_activ                        0
date_end                          0
date_modif_prod                   0
date_renewal                      0
forecast_cons_12m                 0
forecast_cons_year                0
forecast_discount_energy          0
forecast_meter_rent_12m           0
forecast_price_energy_off_peak    0
forecast_price_energy_peak        0
forecast_price_pow_off_peak       0
has_gas                           0
imp_cons                          0
margin_gross_pow_ele              0
margin_net_pow_ele                0
nb_prod_act                       0
net_margin                        0
num_years_antig                   0
origin_up                         0
pow_max                           0
churn                             0
dtype: int64

1월과 12월의 가격 변동을 비교
- 결과값 > 0 => 1월보다 12월 요금이 인상
- 결과값 < 0 => 1월보다 12월 요금이 인하

=> 1년동안 금액 변화량 : price_diff 컬럼 만들기

In [33]:
# 1월의 금액만 추출
price_jan = price_df[price_df['price_date'] == '2015-01-01']

# 12월의 금액만 추출
price_dec = price_df[price_df['price_date'] == '2015-12-01']

# price_data를 제외한 컬럼 리스트만들기
price_cols = [
    'price_off_peak_var', 'price_off_peak_fix',
    'price_peak_var', 'price_peak_fix',
    'price_mid_peak_var', 'price_mid_peak_fix'
]

# 'id'를 인덱스로 만들어 고객 id 별로 12월 - 1월 가격 변동액 확인 : 1년동안의 변화량
price_diff = price_dec.set_index('id')[price_cols] - price_jan.set_index('id')[price_cols]
price_diff.head()

# 인덱스 초기화 -> 파일 두개 합치기 위해
price_diff = price_diff.reset_index()

price_diff

,id,price_off_peak_var,price_off_peak_fix,price_peak_var,price_peak_fix,price_mid_peak_var,price_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916,-0.002302,0.097749,0.003487,0.065166
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779,0.000000,0.000000,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000,0.000000,0.000000,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916,-0.005120,0.097749,0.000763,0.065166
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,-0.050232,-0.335085,-0.038788,-0.400251,-0.022735,-0.432834
16092,fffac626da707b1b5ab11e8431a4d0a2,-0.003778,0.177779,0.000000,0.000000,0.000000,0.000000
16093,fffc0cacd305dd51f316424bbb08d1bd,-0.001760,0.164916,-0.003707,0.099749,-0.007326,0.067166
16094,fffe4f5646aa39c7f97f95ae2679ce64,-0.009391,0.162916,-0.004937,0.097749,0.001029,0.065166


In [34]:
# price_df의 id 기준으로 price_date 제외한 모든 컬럼 중복 제거(중복값은 평균값으로 처리)
mean_price = price_df.groupby('id')[['price_off_peak_var', 'price_peak_var','price_mid_peak_var', 'price_off_peak_fix', 'price_peak_fix', 'price_mid_peak_fix']].mean()
mean_price

,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
id,,,,,,
0002203ffbb812588b632b9e628cc38d,0.124338,0.103794,0.073160,40.701732,24.421038,16.280694
0004351ebdd665e6ee664792efc4fd13,0.146426,0.000000,0.000000,44.385450,0.000000,0.000000
0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.000000,0.000000,45.319710,0.000000,0.000000
0010ee3855fdea87602a5b7aba8e42de,0.118757,0.098292,0.069032,40.647427,24.388455,16.258971
00114d74e963e47177db89bc70108537,0.147926,0.000000,0.000000,44.266930,0.000000,0.000000
...,...,...,...,...,...,...
ffef185810e44254c3a4c6395e6b4d8a,0.138863,0.115125,0.080780,40.896427,24.637456,16.507972
fffac626da707b1b5ab11e8431a4d0a2,0.147137,0.000000,0.000000,44.311375,0.000000,0.000000
fffc0cacd305dd51f316424bbb08d1bd,0.153879,0.129497,0.094842,41.160171,24.895768,16.763569


In [35]:
# client_df를 기준으로 mean_price와 합치기
client_mean_price_df = pd.merge(client_df, mean_price, on='id', how='left')
client_mean_price_df

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,num_years_antig,origin_up,pow_max,churn,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,1,0.124787,0.100749,0.066530,40.942265,22.352010,14.901340
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0,0.149609,0.007124,0.000000,44.311375,0.000000,0.000000
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0,0.170512,0.088421,0.000000,44.385450,0.000000,0.000000
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.200,0,0.151210,0.000000,0.000000,44.400265,0.000000,0.000000
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,19.800,0,0.124174,0.103638,0.072865,40.688156,24.412893,16.275263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14601,18463073fb097fc0ac5d3e040f356987,foosdfpfkusacimwkcsosbicdxkicaua,32270,47940,0,2012-05-24,2016-05-08,2015-05-08,2014-05-26,4648.01,...,4,lxidpiddsbxsbosboudacockeimpuepw,15.000,0,0.144124,0.000000,0.000000,44.370635,0.000000,0.000000
14602,d0a6f71671571ed83b2645d23af6de00,foosdfpfkusacimwkcsosbicdxkicaua,7223,0,181,2012-08-27,2016-08-27,2012-08-27,2015-08-28,631.69,...,3,lxidpiddsbxsbosboudacockeimpuepw,6.000,1,0.106799,0.095406,0.070817,59.015674,36.393379,8.345418
14603,10e6828ddd62cbcf687cb74928c4c2d2,foosdfpfkusacimwkcsosbicdxkicaua,1844,0,179,2012-02-08,2016-02-07,2012-02-08,2015-02-09,190.39,...,4,lxidpiddsbxsbosboudacockeimpuepw,15.935,1,0.124338,0.103794,0.073160,40.701732,24.421038,16.280694
14604,1cf20fd6206d7678d5bcafd28c53b4db,foosdfpfkusacimwkcsosbicdxkicaua,131,0,0,2012-08-30,2016-08-30,2012-08-30,2015-08-31,19.34,...,3,lxidpiddsbxsbosboudacockeimpuepw,11.000,0,0.149609,0.007124,0.000000,44.311375,0.000000,0.000000


In [36]:
# 최종 데이터프레임 
# client_df + mean_price + price_diff 
# client_df : 고객정보
# mean_price : 평균금액
# price_diff : 1년 금액 변동
client_price_df = pd.merge(client_mean_price_df, price_diff, on='id', suffixes=('', '_diff'))
print(client_price_df.shape)
client_price_df.head()

(14606, 38)


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix,price_off_peak_var_diff,price_off_peak_fix_diff,price_peak_var_diff,price_peak_fix_diff,price_mid_peak_var_diff,price_mid_peak_fix_diff
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.066530,40.942265,22.352010,14.901340,0.020057,3.700961,-0.017912,-24.339581,-0.071536,-16.226389
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000,44.311375,0.000000,0.000000,-0.003767,0.177779,0.000000,0.000000,0.000000,0.000000
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000,44.385450,0.000000,0.000000,-0.004670,0.177779,0.000528,0.000000,0.000000,0.000000
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000000,44.400265,0.000000,0.000000,-0.004547,0.177779,0.000000,0.000000,0.000000,0.000000
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.072865,40.688156,24.412893,16.275263,-0.006192,0.162916,-0.002302,0.097749,0.003487,0.065166


In [37]:
# 이탈 비율 확인(0 : 유지 / 1 : 이탈)
client_price_df['churn'].value_counts(normalize=True)

churn
0    0.902848
1    0.097152
Name: proportion, dtype: float64

"에너지 피크 요금이 높을 때 이탈을 많이 한다"라고 가정하였을때
- 에너지 가격 : 고객이 실제 사용한 전력량

In [38]:
client_price_df.groupby('churn')['price_peak_var'].mean()

churn
0    0.051579
1    0.056562
Name: price_peak_var, dtype: float64

"전력 피크 요금이 높을 때 이탈을 많이 한다"라고 가정하였을때
- 전력 가격 : 기본료

In [39]:
# 에너지 피크 요금이 높을 때 이탈을 많이 한다
client_price_df.groupby('churn')['price_peak_fix'].mean()

churn
0     9.274093
1    11.196663
Name: price_peak_fix, dtype: float64

In [40]:
client_price_df['price_diff'] = 

SyntaxError: invalid syntax (1552007568.py, line 1)